In [ ]:
image_name = "NL-HaNA_2.10.50_45_0143.jpg"

In [2]:
import os, json

# Go outside the src directory
os.chdir("..")
current_dir = os.getcwd()
print("Current directory:", current_dir)

Current directory: /Users/sarah_shoilee/codeProjects/stamboekn_KE


### Ground Truth Cell Bounding Box Visualization

In [3]:
### Polygon json Visaulaisation

#!/usr/bin/env python3
"""
Polygon Visualiser
Author: Your Name
Description:
    Reads a polygon JSON file and an image.
    Draws bounding boxes for each cell based on polygon points.
    Annotates each box with its row and column IDs.
"""

import os
import json
import cv2
import numpy as np


class PolygonVisualizer:
    """
    Class to handle polygon parsing and visualisation.
    """

    def __init__(self, polygon_path: str, image_path: str):
        self.polygon_path = polygon_path
        self.image_path = image_path
        self.cells = []

    def load_polygon_data(self):
        """
        Load polygon data from JSON file.
        Each cell includes id, row, col, and points.
        """
        if not os.path.exists(self.polygon_path):
            raise FileNotFoundError(f"Polygon file not found: {self.polygon_path}")

        with open(self.polygon_path, "r", encoding="utf-8") as file:
            try:
                data = json.load(file)
                if not isinstance(data, list):
                    raise ValueError("Polygon JSON must be a list of objects.")
                self.cells = data
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON format: {e}")

    def draw_visualisation(self):
        """
        Draw bounding boxes and annotations on the image.
        """
        if not os.path.exists(self.image_path):
            raise FileNotFoundError(f"Image file not found: {self.image_path}")

        image = cv2.imread(self.image_path)
        if image is None:
            raise ValueError("Failed to load image. Check file format and path.")

        for cell in self.cells:
            points = cell.get("points", [])
            if len(points) >= 4:
                # Convert points to NumPy array for OpenCV
                pts = np.array(points, dtype=np.int32).reshape((-1, 1, 2))

                # Draw polygon
                cv2.polylines(image, [pts], True, (0, 255, 0), 2)

                # Compute text position (top-left corner)
                x, y = points[0]
                label = f"r:{cell.get('row')} c:{cell.get('col')}"
                cv2.putText(image, label, (x + 10, y + 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 0, 0), 2)

        return image


def main():
    """
    Main function to demonstrate polygon visualisation.
    Example usage:
        python polygon_visualiser.py
    """
    print("Starting polygon visualisation demo...\n")

    # Example files (replace with your paths)
    polygon_file = f"data/labels/polygons/{image_name}.polygons.json"  # JSON file with polygon data
    image_file = f"data/images/{image_name}"  # Image file to draw on

    try:
        visualizer = PolygonVisualizer(polygon_file, image_file)
        visualizer.load_polygon_data()
        result_image = visualizer.draw_visualisation()

        # # Display the image
        # cv2.imshow("Polygon Visualisation", result_image)
        # cv2.waitKey(0)
        # cv2.destroyAllWindows()

        # Optionally save the output
        output_path = f"examples/gt_cell_{image_name}"
        cv2.imwrite(output_path, result_image)
        print(f"✅ Visualisation saved to {output_path}")

    except Exception as e:
        error_type = type(e).__name__
        print(f"❌ [ERROR] Failed to process polygon visualisation.")
        print(f"   → Error Type: {error_type}")
        print(f"   → Details: {e}")
        print("   → Suggested Fix: Check file paths and JSON structure.\n")


if __name__ == "__main__":
    main()


Starting polygon visualisation demo...

✅ Visualisation saved to examples/gt_cell_NL-HaNA_2.10.50_45_0143.jpg


### Predicted Cell Bounding Box Visualisation

In [4]:
### PageXML Visualisation
#!/usr/bin/env python3
"""
PageXML Visualiser
Author: Your Name
Description:
    Reads a PageXML file and its associated image.
    Draws bounding boxes for each TableCell based on Coords points.
    Annotates each box with row, col, rowSpan, and colSpan.
"""

import os
import cv2
import numpy as np
import xml.etree.ElementTree as ET


class PageXMLVisualizer:
    """
    Class to handle PageXML parsing and visualisation.
    """

    def __init__(self, pagexml_path: str, image_path: str):
        self.pagexml_path = pagexml_path
        self.image_path = image_path
        self.cells = []

    def parse_pagexml(self):
        """
        Parse PageXML and extract TableCell information.
        Each cell includes coords, row, col, rowSpan, colSpan.
        """
        if not os.path.exists(self.pagexml_path):
            raise FileNotFoundError(f"PageXML file not found: {self.pagexml_path}")

        tree = ET.parse(self.pagexml_path)
        root = tree.getroot()

        # Namespace handling
        ns = {"pc": "http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15"}

        for cell in root.findall(".//pc:TableCell", ns):
            coords_elem = cell.find("pc:Coords", ns)
            coords_points = coords_elem.get("points") if coords_elem is not None else ""
            row = cell.get("row", "")
            col = cell.get("col", "")
            row_span = cell.get("rowSpan", "")
            col_span = cell.get("colSpan", "")

            # Convert coords to list of tuples
            points = []
            if coords_points.strip():
                for pt in coords_points.split():
                    x, y = pt.split(",")
                    points.append((int(float(x)), int(float(y))))

            self.cells.append({
                "points": points,
                "row": row,
                "col": col,
                "rowSpan": row_span,
                "colSpan": col_span
            })

    def draw_visualisation(self):
        """
        Draw bounding boxes and annotations on the image.
        """
        if not os.path.exists(self.image_path):
            raise FileNotFoundError(f"Image file not found: {self.image_path}")

        image = cv2.imread(self.image_path)
        if image is None:
            raise ValueError("Failed to load image. Check file format and path.")

        for cell in self.cells:
            points = cell["points"]
            if len(points) >= 4:
                # Convert points to NumPy array for OpenCV
                pts = np.array(points, dtype=np.int32).reshape((-1, 1, 2))

                # Draw polygon
                cv2.polylines(image, [pts], True, (0, 255, 0), 2)

                # Compute text position (top-left corner)
                x, y = points[0]
                label = f"r:{cell['row']} c:{cell['col']} rs:{cell['rowSpan']} cs:{cell['colSpan']}"
                cv2.putText(image, label, (x + 5, y + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

        return image

In [5]:
image_filename = f"data/images/{image_name}"
pagexml_file = f"data/tables/pagexml/{image_name}.xml"

visualizer = PageXMLVisualizer(pagexml_file, image_filename)
visualizer.parse_pagexml()
result_image = visualizer.draw_visualisation()
# Display the image
# cv2.imshow("PageXML Visualisation", result_image)
# cv2.waitKey()
# cv2.destroyAllWindows()

# save the  visualisation
output_path = f"examples/pred_cell_{image_name}"
cv2.imwrite(output_path, result_image)
print(f"✅ Visualisation saved to {output_path}")

✅ Visualisation saved to examples/pred_cell_NL-HaNA_2.10.50_45_0143.jpg


### Compute maP

In [6]:
import os
from src.metrics import compute_mAP
from src.utils import pagexml_to_html 



OUTPUT_HTML_DIR = "examples"
DATA_DIR = "data/tables/pagexml"
GT_POLYGON_DIR = "data/labels/polygons"

pagexml_file = os.path.join(DATA_DIR, f"{image_name}.xml")
output_html_file = os.path.join(OUTPUT_HTML_DIR, f"{image_name}.html")
pagexml_to_html(pagexml_file, output_html_file)

# --- Compute mAP ---
gt_file = os.path.join(GT_POLYGON_DIR, f"{image_name}.polygons.json")
pred_file = pagexml_file
mAP = compute_mAP(gt_file, pred_file)

mAP@0.1-1.0: 0.3250
Matches: 12, Unmatched GT: 0


## Information Extraction

### information extraction from ground truth html

In [12]:
html_file = f"data/tables/html/{image_name}.html"
with open(html_file, encoding="utf-8") as f:
    pred_html = f.read()

In [13]:
import shutil, json
from src.person_info_extraction_ontogpt import extract_person_info as information_extractor
from experiment_1 import parse_html_table

In [14]:
json_out_path = os.path.join('examples', f"{image_name}.json")

TEMP_DIR = "examples/temp_persons"
os.makedirs(TEMP_DIR, exist_ok=True)

SCHEMA_PATH = "data/schema/personbasicinfo.yaml"
LLM_MODEL = "ollama/llama3"

logical_rows = parse_html_table(pred_html)

try:
    for i, row in enumerate(logical_rows):
        print(f"Processing row {i+1}/{len(logical_rows)}")
        temp_file = f"person_{i}.json"
        try:
            information_extractor(i, row, schema_path=SCHEMA_PATH, json_output=os.path.join(TEMP_DIR, temp_file), temp_dir=TEMP_DIR, llm_model=LLM_MODEL)
        except Exception as e:
            print(f" ❌ Error processing row {i}: {e}")
            traceback.print_exc()
            continue

    persons = []

    for filename in os.listdir(TEMP_DIR):
        if filename.endswith(".json") and filename.startswith("person_"):
            with open(os.path.join(TEMP_DIR, filename), 'r', encoding='utf-8') as f:
                data = json.load(f)
                # each temp file is expected to be {'persons': [...]}
                person = data.get("persons", [])
                if isinstance(persons, list):
                    persons.extend(person)
                elif person:
                    persons.append(persons)

    with open(json_out_path, 'w', encoding='utf-8') as f:
        json.dump({"persons": persons}, f, indent=2, ensure_ascii=False)   
finally: 
    shutil.rmtree(TEMP_DIR, ignore_errors=True)

Processing row 1/4
Running OntoGPT...

Processing row 2/4
Running OntoGPT...

Processing row 3/4
Running OntoGPT...

Processing row 4/4
Running OntoGPT...



In [6]:
from src.metrics import infomration_extraction_precision_recall
json_out_path = os.path.join('data/json', f"{image_name}.json")

with open(f"data/labels/info/{image_name.replace('.jpg', '.json')}", encoding="utf-8") as f:
        gt_info = json.load(f)
with open(json_out_path, encoding="utf-8") as f:
    pred_info = json.load(f)

# info_sim = best_match_similarity(gt_info.get("persons", []), pred_info.get("persons", []))
precision, recall, f1_score = infomration_extraction_precision_recall(
    pred_info.get("persons", []), gt_info.get("persons", []), threshold=0.4
)

print(f"Information Extraction - \nPrecision: {precision:.4f}, \nRecall: {recall:.4f}, \nF1-score: {f1_score:.4f}")

Matched pairs [(np.int64(0), np.int64(0)), (np.int64(1), np.int64(1)), (np.int64(2), np.int64(2)), (np.int64(3), np.int64(3))] with similarities [np.float64(0.511239554096697), np.float64(0.0), np.float64(0.0), np.float64(0.0)]
Predicted fields: 6
Ground Truth fields: 7
Correctly extracted fields (TP): 4
Information Extraction - 
Precision: 0.6667, 
Recall: 0.5714, 
F1-score: 0.6154
